# Robustness evaluation and controlled rerun

This notebook executes the post-Task-3 milestone. It evaluates existing Stage A and RINE checkpoints, retrains controlled RINE across seeds 42/43/44, then tests retained frequency and Lab fusion candidates. It never reads `final_test`; Tasks 9 and 10 remain downstream.

Run cells top to bottom in a single session. Each `make`/script call prints its own stdout/stderr and raises with a clear message on failure, so a failed cell should not be re-run blindly — read the printed error first.

## 0. Clone/refresh the repository and mount Drive

In [50]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
REPO_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'

if PROJECT.is_dir():
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT, capture_output=True, text=True)
else:
    result = subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'repository clone/pull failed'

Already up to date.




In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
import json

ARTIFACTS = PROJECT / 'artifacts'
ROBUSTNESS = ARTIFACTS / 'robustness'
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
DURABLE_ROBUSTNESS_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts/robustness')
SEEDS = (42, 43, 44)

assert PROJECT.is_dir(), PROJECT
assert PRIOR.is_dir(), f'Drive artifacts root not found: {PRIOR}'
print({'project': str(PROJECT), 'robustness': str(ROBUSTNESS), 'prior': str(PRIOR)})


def run_make(*args, cwd=PROJECT):
    result = subprocess.run(['make', *args], cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"make {' '.join(args)} failed with code {result.returncode}")
    return result


def run_script(*args, cwd=PROJECT):
    result = subprocess.run(list(args), cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(args)} failed with code {result.returncode}")
    return result

{'project': '/content/cya-techjam26', 'robustness': '/content/cya-techjam26/artifacts/robustness', 'prior': '/content/drive/MyDrive/cya-techjam26/artifacts'}


## 1. Install and preflight
Installs Colab dependencies, then runs the bootstrap smoke check (tolerates the optional `c2pa-python` package, which the robustness pipeline never imports) followed by the robustness-specific unit tests.

In [53]:
run_make('install-colab')

python -m pip install -r requirements-colab.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 109.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
python -m pip install -e . --no-deps
Obtaining file:///content/cya-techjam26
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'd

CompletedProcess(args=['make', 'install-colab'], returncode=0, stdout='python -m pip install -r requirements-colab.txt\nRequirement already satisfied: transformers<6,>=4.44 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 3)) (5.15.1)\nRequirement already satisfied: accelerate<2,>=0.33 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 4)) (1.14.0)\nRequirement already satisfied: safetensors<1,>=0.4 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 5)) (0.8.0)\nRequirement already satisfied: numpy<3,>=1.26 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 6)) (2.1.3)\nRequirement already satisfied: scipy<2,>=1.13 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 7)) (1.16.3)\nRequirement already satisfied: pandas<4,>=2.2 in /usr/local/lib/python3.13/dist-packages (from -r requirements-colab.txt (line 8)) (2.2.3)\nRequirement already 

In [54]:
run_make('smoke-bootstrap')
run_make('robustness-test')

python scripts/smoke_check.py --config configs/colab.json --allow-missing-dependencies
{
  "accelerator": {
    "cuda_available": true,
    "device": "Tesla T4",
    "torch_available": true
  },
  "config": "ok",
  "imports": {
    "Pillow": "ok",
    "c2pa-python": "ok",
    "numpy": "ok",
    "opencv-python-headless": "ok",
    "pandas": "ok",
    "scikit-image": "ok",
    "scikit-learn": "ok",
    "scipy": "ok",
    "torch": "ok",
    "torchvision": "ok",
    "transformers": "ok"
  },
  "model": "openai/clip-vit-large-patch14-336",
  "runtime": "google_colab",
  "schema_version": 1
}


PYTHONPATH=src python -m unittest tests.test_robustness_training tests.test_evaluation tests.test_controlled_sampler -v

test_bank_rejects_missing_cells_crossed_labels_and_final_test (tests.test_robustness_training.RobustnessTrainingTests.test_bank_rejects_missing_cells_crossed_labels_and_final_test) ... ok
test_complete_bank_is_ordered_and_preserves_all_cells (tests.test_robustness_training.Robustnes

CompletedProcess(args=['make', 'robustness-test'], returncode=0, stdout='PYTHONPATH=src python -m unittest tests.test_robustness_training tests.test_evaluation tests.test_controlled_sampler -v\n', stderr='test_bank_rejects_missing_cells_crossed_labels_and_final_test (tests.test_robustness_training.RobustnessTrainingTests.test_bank_rejects_missing_cells_crossed_labels_and_final_test) ... ok\ntest_complete_bank_is_ordered_and_preserves_all_cells (tests.test_robustness_training.RobustnessTrainingTests.test_complete_bank_is_ordered_and_preserves_all_cells) ... ok\ntest_controlled_epoch_selects_only_complete_cached_views (tests.test_robustness_training.RobustnessTrainingTests.test_controlled_epoch_selects_only_complete_cached_views) ... ok\ntest_controlled_rine_selects_a_real_50_50_checkpoint (tests.test_robustness_training.RobustnessTrainingTests.test_controlled_rine_selects_a_real_50_50_checkpoint) ... ok\ntest_feature_bank_keeps_magnitude_residual_and_lab_but_drops_phase_rgb (tests.test_

## 2. Stage the raw SID dataset locally
Copies all 20,000 source images from the `hackathon_data` Drive shortcut into local `/content` scratch disk. Resumable: already-copied files (matched by size) are skipped, so re-running after an interruption is safe.

In [55]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil

source_root = Path('/content/drive/MyDrive/hackathon_data/raw/sid_set')
source_images = source_root / 'images'
assert source_root.is_dir(), (
    f'Drive source not found: {source_root}. '
    'Add a Drive shortcut to the shared dataset folder named exactly "hackathon_data" under My Drive.'
)

destination_root = Path('/content/hackathon_data/raw/sid_set')
destination_images = destination_root / 'images'
destination_images.mkdir(parents=True, exist_ok=True)

shutil.copy2(source_root / 'labels.csv', destination_root / 'labels.csv')

source_files = [path for path in source_images.iterdir() if path.is_file()]
pending = []
for source_file in source_files:
    destination_file = destination_images / source_file.name
    if not destination_file.exists() or destination_file.stat().st_size != source_file.stat().st_size:
        pending.append(source_file)

print('Total source files:', len(source_files))
print('Already complete:', len(source_files) - len(pending))
print('Remaining:', len(pending))


def copy_one(source_file):
    destination_file = destination_images / source_file.name
    temporary_file = destination_images / f'{source_file.name}.part'
    shutil.copy2(source_file, temporary_file)
    temporary_file.replace(destination_file)
    return source_file.name


errors = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(copy_one, source_file): source_file for source_file in pending}
    for completed, future in enumerate(as_completed(futures), start=1):
        try:
            future.result()
        except Exception as error:
            errors.append((futures[future].name, str(error)))
        if completed % 500 == 0 or completed == len(futures):
            print(f'Progress: {completed}/{len(futures)} processed; errors: {len(errors)}')

final_count = sum(1 for path in destination_images.iterdir() if path.is_file() and not path.name.endswith('.part'))
print('\nFinal verification')
print('Images:', final_count)
print('Errors:', len(errors))
print('Dataset complete:', final_count == 20_000)

if errors:
    print('First errors:', errors[:10])

assert not errors, f'{len(errors)} files failed to copy'
assert final_count == 20_000, f'Expected 20,000 images, found {final_count}'
print('PASS: raw SID dataset staged locally.')

Total source files: 20000
Already complete: 0
Remaining: 20000
Progress: 500/20000 processed; errors: 0
Progress: 1000/20000 processed; errors: 0
Progress: 1500/20000 processed; errors: 0
Progress: 2000/20000 processed; errors: 0
Progress: 2500/20000 processed; errors: 0
Progress: 3000/20000 processed; errors: 0
Progress: 3500/20000 processed; errors: 0
Progress: 4000/20000 processed; errors: 0
Progress: 4500/20000 processed; errors: 0
Progress: 5000/20000 processed; errors: 0
Progress: 5500/20000 processed; errors: 0
Progress: 6000/20000 processed; errors: 0
Progress: 6500/20000 processed; errors: 0
Progress: 7000/20000 processed; errors: 0
Progress: 7500/20000 processed; errors: 0
Progress: 8000/20000 processed; errors: 0
Progress: 8500/20000 processed; errors: 0
Progress: 9000/20000 processed; errors: 0
Progress: 9500/20000 processed; errors: 0
Progress: 10000/20000 processed; errors: 0
Progress: 10500/20000 processed; errors: 0
Progress: 11000/20000 processed; errors: 0
Progress: 1

## 3. Regenerate the matched-clean (fixed-Q96) parents
Task 2's derived JPEGs were never synced to Drive, only its manifests/reports. Regenerate them deterministically (seed 42, `fixed_q96` policy) from the raw images just staged, then confirm the regenerated manifest is byte-identical to Drive's stored one before trusting it.

In [57]:
import csv

with open(PRIOR / 'task2' / 'fixed_q96_manifest.csv') as f:
    drive_rows = list(csv.DictReader(f))
with open(ARTIFACTS / 'task2' / 'fixed_q96_manifest_regenerated.csv') as f:
    regen_rows = list(csv.DictReader(f))

print('row counts:', len(drive_rows), len(regen_rows))

drive_by_id = {r['sample_id']: r for r in drive_rows}
regen_by_id = {r['sample_id']: r for r in regen_rows}

print('sample_id sets equal:', set(drive_by_id) == set(regen_by_id))

diffs = []
for sample_id in sorted(set(drive_by_id) & set(regen_by_id)):
    a, b = drive_by_id[sample_id], regen_by_id[sample_id]
    for key in a:
        if key in ('image_path', 'clean_image_path', 'source_path'):
            continue  # expected to differ across environments/output roots
        if a.get(key) != b.get(key):
            diffs.append((sample_id, key, a.get(key), b.get(key)))

print('field diffs (excluding path columns):', len(diffs))
for d in diffs[:15]:
    print(d)


row counts: 2000 2000
sample_id sets equal: True
field diffs (excluding path columns): 0


In [56]:
import hashlib

run_script(
    'python', 'scripts/build_matched_clean.py',
    '--source-manifest', str(PRIOR / 'task2' / 'source_manifest_split.csv'),
    '--output-root', str(ARTIFACTS / 'task2' / 'matched_candidates'),
    '--output-manifest', str(ARTIFACTS / 'task2' / 'fixed_q96_manifest_regenerated.csv'),
    '--report', str(ARTIFACTS / 'task2' / 'fixed_q96_report_regenerated.json'),
    '--policy', 'fixed_q96', '--seed', '42', '--limit-per-label', '1000',
)

drive_hash = hashlib.sha256((PRIOR / 'task2' / 'fixed_q96_manifest.csv').read_bytes()).hexdigest()
regen_hash = hashlib.sha256((ARTIFACTS / 'task2' / 'fixed_q96_manifest_regenerated.csv').read_bytes()).hexdigest()
match = {'drive_manifest_sha256': drive_hash, 'regenerated_manifest_sha256': regen_hash, 'match': drive_hash == regen_hash}
print(match)
assert match['match'], 'Regenerated fixed-Q96 manifest does not match the Drive copy; stop and investigate before continuing.'

{
  "encoder_version": "Pillow-11.3.0",
  "image_count": 2000,
  "label_counts": {
    "ai_generated": 1000,
    "authentic": 1000
  },
  "limit_per_label": 1000,
  "metadata_policy": "strip_exif",
  "output_manifest": "/content/cya-techjam26/artifacts/task2/fixed_q96_manifest_regenerated.csv",
  "output_manifest_sha256": "61169374947286a0b25d855ea7039d28a49ed3f1eeb18f2d6ae7e9a501679782",
  "policy": "fixed_q96",
  "quality_counts": {
    "96": 2000
  },
  "resize_applied": false,
  "seed": 42,
  "source_manifest": "/content/drive/MyDrive/cya-techjam26/artifacts/task2/source_manifest_split.csv",
  "source_manifest_sha256": "a3dc5e48330124daa5d0ca13a01075b010916f8c698924116dc4f2c9f4eec630",
  "subsampling": "4:4:4"
}


{'drive_manifest_sha256': 'aee4bd2e16fec2208cea4a7834a2c6b6086c5edfb9c5c64df21f54cc89ff3ef2', 'regenerated_manifest_sha256': '61169374947286a0b25d855ea7039d28a49ed3f1eeb18f2d6ae7e9a501679782', 'match': False}


AssertionError: Regenerated fixed-Q96 manifest does not match the Drive copy; stop and investigate before continuing.

## 4. Prepare independent robustness views
Writes a development-only matched-clean parent manifest, materializes all 14 Task 3 cells directly from those parents, validates the complete bank, and creates a combined feature-extraction manifest.

In [58]:
run_make('robustness-prepare', f'TASK2_SELECTED_MANIFEST={PRIOR}/task2/fixed_q96_manifest.csv')

python scripts/prepare_robustness_manifest.py --input-manifest /content/drive/MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv --output-manifest artifacts/robustness/manifests/dev_clean_manifest.csv --report artifacts/robustness/manifests/clean_manifest_report.json
{
  "allowed_splits": [
    "seed_train",
    "selection_val"
  ],
  "counts": {
    "seed_train:ai_generated": 618,
    "seed_train:authentic": 607,
    "selection_val:ai_generated": 89,
    "selection_val:authentic": 76
  },
  "final_test_rows": 0,
  "input_manifest": "/content/drive/MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv",
  "input_manifest_sha256": "aee4bd2e16fec2208cea4a7834a2c6b6086c5edfb9c5c64df21f54cc89ff3ef2",
  "output_manifest": "/content/cya-techjam26/artifacts/robustness/manifests/dev_clean_manifest.csv",
  "output_manifest_sha256": "656d6c4879eba44da43d6d71f3919d379559017d82ede404c20df8a909fb3560",
  "row_count": 1390
}
python scripts/materialize_transforms.py --input-manifest 

CompletedProcess(args=['make', 'robustness-prepare', 'TASK2_SELECTED_MANIFEST=/content/drive/MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv'], returncode=0, stdout='python scripts/prepare_robustness_manifest.py --input-manifest /content/drive/MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv --output-manifest artifacts/robustness/manifests/dev_clean_manifest.csv --report artifacts/robustness/manifests/clean_manifest_report.json\n{\n  "allowed_splits": [\n    "seed_train",\n    "selection_val"\n  ],\n  "counts": {\n    "seed_train:ai_generated": 618,\n    "seed_train:authentic": 607,\n    "selection_val:ai_generated": 89,\n    "selection_val:authentic": 76\n  },\n  "final_test_rows": 0,\n  "input_manifest": "/content/drive/MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv",\n  "input_manifest_sha256": "aee4bd2e16fec2208cea4a7834a2c6b6086c5edfb9c5c64df21f54cc89ff3ef2",\n  "output_manifest": "/content/cya-techjam26/artifacts/robustness/manifests/dev_cle

## 5. Evaluate existing clean-trained checkpoints
These runs update no weights and establish the pre-retraining robustness baseline.

In [59]:
for seed in SEEDS:
    stage_a = PRIOR / 'task4/fixed_q96' / f'seed_{seed}' / 'best_clean.pt'
    rine = PRIOR / 'task6/fixed_q96' / f'seed_{seed}' / 'best_clean.pt'
    assert stage_a.is_file(), stage_a
    assert rine.is_file(), rine
    run_make('robustness-stage-a-evaluate', f'ROBUSTNESS_SEED={seed}', f'STAGE_A_CHECKPOINT={stage_a}')
    run_make('robustness-rine-evaluate', f'ROBUSTNESS_SEED={seed}', f'RINE_CHECKPOINT={rine}')

python scripts/run_robustness.py evaluate-stage-a --clean-manifest artifacts/robustness/manifests/dev_clean_manifest.csv --transform-manifest artifacts/robustness/manifests/transform_manifest.csv --checkpoint /content/drive/MyDrive/cya-techjam26/artifacts/task4/fixed_q96/seed_42/best_clean.pt --output-root artifacts/robustness --seed 42 --physical-batch-size 4
{
  "all_records": {
    "accuracy": 0.9252525252525252,
    "ai_generated_accuracy": 0.8719101123595505,
    "authentic_accuracy": 0.987719298245614,
    "confusion_matrix": {
      "true_ai_generated_pred_ai_generated": 1164,
      "true_ai_generated_pred_authentic": 171,
      "true_authentic_pred_ai_generated": 14,
      "true_authentic_pred_authentic": 1126
    },
    "ece": 0.12289799948364045,
    "false_negative_rate": 0.12808988764044943,
    "false_positive_rate": 0.012280701754385965,
    "sample_count": 2475,
    "threshold": 0.5
  },
  "bootstrap_intervals": {
    "clean_accuracy": {
      "confidence": 0.95,
      "

## 6. Controlled RINE rerun
The CLIP tower stays frozen. Only the RINE layer weighting and binary head train under the balanced clean-or-one-transform sampler.

In [60]:
for seed in SEEDS:
    run_make('robustness-rine-train', f'ROBUSTNESS_SEED={seed}')

python scripts/run_robustness.py train-controlled-rine --clean-manifest artifacts/robustness/manifests/dev_clean_manifest.csv --transform-manifest artifacts/robustness/manifests/transform_manifest.csv --output-root artifacts/robustness --seed 42 --physical-batch-size 4
{
  "accumulation_steps": 8,
  "best_layer_importance": [
    0.16542915999889374,
    0.227853924036026,
    0.30267757177352905,
    0.30403926968574524
  ],
  "best_values": {
    "clean": 1.0,
    "robustness": 0.996969696969697,
    "selection_score": 0.9984848484848485
  },
  "effective_batch_size": 32,
  "epoch_size": 1225,
  "epochs_completed": 17,
  "history": [
    {
      "clean_accuracy": 0.9515151515151515,
      "epoch": 1,
      "layer_importance": [
        0.24400997161865234,
        0.25173866748809814,
        0.25259605050086975,
        0.25165536999702454
      ],
      "robustness_mean_accuracy": 0.9510822510822511,
      "selection_score": 0.9512987012987013,
      "training_loss": 0.573297602637

## 7. Extract retained auxiliary features
Frequency extraction excludes phase during fusion. Auxiliary extraction supplies Lab; RGB, PRNU, and optics are not fused by this milestone.

In [61]:
run_make('robustness-frequency-extract')
run_make('robustness-lab-extract')

python scripts/extract_frequency_features.py --manifest artifacts/robustness/manifests/combined_manifest.csv --output artifacts/robustness/features/frequency_features.csv --report artifacts/robustness/features/frequency_extraction_report.json --cache-root /content/robustness_frequency_cache --matching-policy fixed_q96 --workers 4
{
  "configuration": {
    "angular_bins": 12,
    "dct_bins": 16,
    "extractor_version": "frequency-v1",
    "max_analysis_size": 1024,
    "phase_bins": 12,
    "radial_bins": 24,
    "workers": 4
  },
  "error_counts": {},
  "extractor_version": "frequency-v1",
  "feature_count": 92,
  "feature_families": {
    "dct_high_energy_ratio": "magnitude",
    "dct_radial_00": "magnitude",
    "dct_radial_01": "magnitude",
    "dct_radial_02": "magnitude",
    "dct_radial_03": "magnitude",
    "dct_radial_04": "magnitude",
    "dct_radial_05": "magnitude",
    "dct_radial_06": "magnitude",
    "dct_radial_07": "magnitude",
    "dct_radial_08": "magnitude",
    "d

CompletedProcess(args=['make', 'robustness-lab-extract'], returncode=0, stdout='python scripts/extract_auxiliary_features.py --manifest artifacts/robustness/manifests/combined_manifest.csv --output artifacts/robustness/features/auxiliary_features.csv --report artifacts/robustness/features/auxiliary_extraction_report.json --cache-root /content/robustness_auxiliary_cache --matching-policy fixed_q96 --families color\n{\n  "configuration": {\n    "ca_min_edge_fraction": 0.02,\n    "ca_scale_limit": 0.006,\n    "ca_scale_steps": 13,\n    "color_window_size": 64,\n    "extractor_version": "auxiliary-v1",\n    "low_variance_epsilon": 0.0001,\n    "max_analysis_size": 1024,\n    "max_eligibility_rate_gap": 0.05,\n    "optics_max_analysis_size": 512,\n    "optics_min_dimension": 256,\n    "prnu_block_size": 64,\n    "prnu_denoise_sigma": 1.0,\n    "prnu_min_dimension": 128,\n    "workers": 4\n  },\n  "coverage_counts": {\n    "seed_train:ai_generated:ca:eligible": 0,\n    "seed_train:ai_generat

## 8. Train individual fusion candidates
Each candidate freezes its controlled-RINE parent and trains only the auxiliary projection and fusion head.

In [62]:
for variant in ('frequency', 'lab'):
    for seed in SEEDS:
        run_make('robustness-fusion-train', f'ROBUSTNESS_FUSION_VARIANT={variant}', f'ROBUSTNESS_SEED={seed}')

python scripts/run_robustness_fusion.py --variant frequency --clean-manifest artifacts/robustness/manifests/dev_clean_manifest.csv --transform-manifest artifacts/robustness/manifests/transform_manifest.csv --parent-checkpoint artifacts/robustness/train-controlled-rine/seed_42/best_50_50.pt --frequency-table artifacts/robustness/features/frequency_features.csv --auxiliary-table artifacts/robustness/features/auxiliary_features.csv --output-root artifacts/robustness --seed 42 --physical-batch-size 4
{
  "best_selection_score": 0.9982683982683982,
  "epochs_completed": 5,
  "feature_names": [
    "frequency:fft_radial_00",
    "frequency:fft_radial_01",
    "frequency:fft_radial_02",
    "frequency:fft_radial_03",
    "frequency:fft_radial_04",
    "frequency:fft_radial_05",
    "frequency:fft_radial_06",
    "frequency:fft_radial_07",
    "frequency:fft_radial_08",
    "frequency:fft_radial_09",
    "frequency:fft_radial_10",
    "frequency:fft_radial_11",
    "frequency:fft_radial_12",
 

## 9. Apply retention gates
A candidate is retained only if its mean 50/50 score strictly improves and neither mean class accuracy regresses by more than one percentage point.

In [63]:
decisions = {}
for variant in ('frequency', 'lab'):
    output = ROBUSTNESS / 'reports' / variant
    run_script(
        'python', 'scripts/compare_robustness_candidate.py',
        '--parent-root', str(ROBUSTNESS / 'train-controlled-rine'),
        '--candidate-root', str(ROBUSTNESS / f'rine_{variant}'),
        '--candidate-name', f'rine_{variant}',
        '--output', str(output),
    )
    decisions[variant] = json.loads((output / 'retention_decision.json').read_text())['decision']
print(decisions)

{
  "candidate": "rine_frequency",
  "decision": "reject",
  "final_test_read": false,
  "max_per_class_accuracy_regression": 0.01,
  "mean_ai_generated_delta": -0.48988764044943817,
  "mean_authentic_delta": -0.44181286549707605,
  "mean_candidate_score": 0.5215007215007215,
  "mean_parent_score": 0.9981240981240981,
  "mean_score_delta": -0.47662337662337667,
  "required_cell_count": 14,
  "seed_results": [
    {
      "ai_generated_delta": 0.001498127340823996,
      "authentic_delta": -0.0026315789473684292,
      "candidate_clean": 1.0,
      "candidate_robustness": 0.9965367965367965,
      "candidate_score": 0.9982683982683982,
      "candidate_worst_cell": "noise_sigma_0.05",
      "candidate_worst_cell_accuracy": 0.9878787878787879,
      "parent_clean": 1.0,
      "parent_robustness": 0.996969696969697,
      "parent_score": 0.9984848484848485,
      "score_delta": -0.0002164502164503368,
      "seed": 42
    },
    {
      "ai_generated_delta": -0.8659176029962546,
      "au

## 10. Conditional combined fusion
Run the combined candidate only if both individual additions passed. Otherwise controlled RINE is the pre-Task-9 handoff.

In [64]:
if decisions == {'frequency': 'retain', 'lab': 'retain'}:
    for seed in SEEDS:
        run_make('robustness-fusion-train', 'ROBUSTNESS_FUSION_VARIANT=frequency_lab', f'ROBUSTNESS_SEED={seed}')
    combined_output = ROBUSTNESS / 'reports' / 'frequency_lab'
    run_script(
        'python', 'scripts/compare_robustness_candidate.py',
        '--parent-root', str(ROBUSTNESS / 'train-controlled-rine'),
        '--candidate-root', str(ROBUSTNESS / 'rine_frequency_lab'),
        '--candidate-name', 'rine_frequency_lab',
        '--output', str(combined_output),
    )
    print(json.loads((combined_output / 'retention_decision.json').read_text())['decision'])
else:
    print('Combined candidate skipped:', decisions)

Combined candidate skipped: {'frequency': 'reject', 'lab': 'reject'}


## 11. Durable sync
Copies completed run directories and reports to Drive at
[`cya-techjam26/artifacts`](https://drive.google.com/drive/folders/1uv0sa041-6N-Vg8tdtb5in0GgWBR-SFz), mirroring
`artifacts/robustness` under the same durable root the other tasks use. Review completion markers first; do not
move or delete the local copy while a run may still be in progress.

In [65]:
import shutil

def sync_tree(local_root: Path, remote_root: Path):
    remote_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    for local_path in local_root.rglob('*'):
        relative = local_path.relative_to(local_root)
        remote_path = remote_root / relative
        if local_path.is_dir():
            remote_path.mkdir(parents=True, exist_ok=True)
            continue
        if not remote_path.exists() or remote_path.stat().st_size != local_path.stat().st_size:
            remote_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, remote_path)
            copied += 1
    return copied


assert ROBUSTNESS.is_dir(), f'Nothing to sync yet: {ROBUSTNESS}'
copied = sync_tree(ROBUSTNESS, DURABLE_ROBUSTNESS_ROOT)
print(f'Synced {copied} new/changed files to {DURABLE_ROBUSTNESS_ROOT}')

Synced 19588 new/changed files to /content/drive/MyDrive/cya-techjam26/artifacts/robustness
